* 학년: 
* 반:
* 번호:
* 이름:

# 선택 활동 1: 우리나라의 지진 발생 현황을 분석해 보기

최근 모로코(규모 6.8), 튀르키예(규모 7.8) 지진으로 수천에서 수만 명의 사망자를 발생하였다. 우리나라도 2016년, 2017년에 연이은 큰 규모의 지진으로 지진 안전지대라고 할 수 없다.

## 단계 0: 준비 (라이브러리 설치)

In [ ]:
import importlib, sys, subprocess

packages = [
    ('pandas', 'pandas'),
    ('numpy', 'numpy'),
    ('sklearn', 'scikit-learn'),
    ('matplotlib.pyplot', 'matplotlib'),
    ('seaborn', 'seaborn'),
]

for module_name, pip_name in packages:
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pip_name])

from sklearn import set_config
set_config(display="text")

print('라이브러리 준비 완료')

## 단계 1: 문제 정의하기
우리나라 지진 발생 현황을 분석해 보자.


## 단계 2: 데이터 수집하기

① 기상자료개방포털 사이트에 접속한다.

> 30일 이상 데이터를 다운로드하기 위해 로그인이 필요하다.

https://data.kma.go.kr

② `[데이터]-[지진화산]-[지진화산 특·정보]-지진정보`에서 최대 120개월의 지진 데이터를 조회한 후 파일을 csv로 저장한다.

③ 데이터를 직접 수집한 경우, 저장한 csv 파일을 utf-8로 변환해서 사용해야 한다.

> 구글 스프레드시트에서 데이터 파일을 열고 `[파일]-[다운로드]-[쉼표로 구분된 값(.csv)]`으로 다시 저장한다. (파일명: earthquake)

④ 데이터를 직접 수집하지 않은 경우에는 주어진 링크에 들어가서 `earthquake.csv`를 다운로드한다.

https://bit.ly/earthquake_data


## 단계 3: 데이터 전처리 및 탐색적 데이터 분석하기

① 코드 작성 시 한글이 깨지는 현상을 방지하기 위해 아래 라이브러리를 설치한다. (교과서의 koreanize-matplotlib이 동작하지 않는 경우가 있어 호환성을 위해 다른 코드를 사용함)

In [ ]:
import platform
import matplotlib.pyplot as plt

def set_korean_font():
    system = platform.system()
    
    # Colab 환경 감지
    try:
        import google.colab
        is_colab = True
    except ImportError:
        is_colab = False

    if is_colab:
        import subprocess
        subprocess.run(['apt-get', '-qq', '-y', 'install', 'fonts-nanum'])
        import matplotlib.font_manager as fm
        fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
        plt.rcParams['font.family'] = 'NanumGothic'
    elif system == 'Windows':
        plt.rcParams['font.family'] = 'Malgun Gothic'
    elif system == 'Darwin':
        plt.rcParams['font.family'] = 'AppleGothic'
    elif system == 'Linux':
        plt.rcParams['font.family'] = 'NanumGothic'

    plt.rcParams['axes.unicode_minus'] = False

set_korean_font()

② 지진 데이터를 불러와 데이터 프레임을 생성한다.

**❓ 질문**
- 이 데이터에는 어떤 속성(열)들이 있는가?

  → ( )


③ 지진 데이터의 결측치를 확인한다.

**❓ 질문**
- 결측치가 있는 속성은 무엇이고, 몇 개인가?

  → ( )


④ 행 단위로 결측치를 확인해 삭제한다.

In [ ]:
# axis=0: 행 단위, inplace=True: 원본 데이터 프레임 변경

⑤ `시간` 속성에 있는 일시에서 연도만 추출한다.

**💡 참고: `시간`에서 `연도` 뽑아내기**

### 1. `시간` 열 선택하기

- **문법 설명:** 데이터 프레임 뒤에 대괄호 `[]`를 쓰고, 그 안에 열 이름을 넣으면 해당 열을 선택할 수 있다.
- **사용 형식:** `df['열 이름']`
- **이번 활동:** 날짜와 시간이 들어 있는 `시간` 열을 선택한다.

### 2. datetime 형식으로 바꾸기

- **API 설명:** `pd.to_datetime()`은 글자 형태의 날짜·시간 데이터를 pandas의 datetime 자료형으로 바꾸어 준다.
- **사용 형식:** `pd.to_datetime(변환할 데이터)`
- **이번 활동:** 1번에서 선택한 `시간` 열을 괄호 안에 넣어 datetime으로 바꾼다.

> `2016-09-12 20:32:54`(글자) → `2016-09-12 20:32:54`(datetime)  
> 보이는 모습은 같지만, datetime으로 바꾸면 연·월·일·시간을 각각 다룰 수 있다.

### 3. 날짜 요소에 접근하기

- **문법 설명:** 여러 행으로 이루어진 datetime 열에서 날짜 전용 기능을 사용할 때는 `.dt`를 붙인다.
- **사용 형식:** `datetime으로 바꾼 열.dt`
- **이번 활동:** 2번의 datetime 결과 뒤에 `.dt`를 붙여 날짜 요소에 접근한다.

### 4. 연도 뽑아내기

- **문법 설명:** `.dt` 뒤에 필요한 날짜 요소의 영어 이름을 붙이면 각 행에서 해당 값을 뽑아낸다.

| 날짜 요소 | 영어 이름 | 결과 예 |
|:---:|:---:|:---:|
| 연도 | `.year` | 2016 |
| 월 | `.month` | 9 |
| 일 | `.day` | 12 |
| 시간 | `.hour` | 20 |

- **이번 활동:** 연도에 해당하는 영어 이름을 `.dt` 뒤에 붙인다.

### 5. `연도` 열에 저장하기

- **문법 설명:** `=`의 오른쪽에서 만든 결과를 왼쪽의 열에 저장한다. 해당 열이 없으면 새로 생성된다.
- **사용 형식:** `df['저장할 열 이름'] = 계산한 결과`
- **이번 활동:** 4번에서 뽑아낸 결과를 `df['연도']`에 저장한다.

In [ ]:
# pd.to_datetime : 날짜 및 시간 정보를 가진 데이터를 pandas의 datetime 형식으로 변환하기
# dt.year : datetime 타입에서 연도만 추출하기

⑥ 연도에 따른 지진 규모를 산점도로 시각화하여 가장 큰 지진이 일어난 연도를 살펴보자.

1. `seaborn`을 `sns`라는 이름으로 불러온다.
2. 연도를 x축, 규모를 y축으로 하는 산점도를 그린다.

   - **기본 형식:** `sns.scatterplot(x='x축 열 이름', y='y축 열 이름', data=데이터 프레임 변수)`
   - x축 열 이름에는 `연도`, y축 열 이름에는 `규모`, 데이터 프레임 변수에는 `df`를 지정한다.


In [ ]:
# seaborn 라이브러리 가져오기
# 산점도 그래프


**❓ 질문**
- 2014년부터 지금까지, 가장 큰 규모의 지진이 일어난 해는 언제인가?  → ( )년
- 그 지진은 언제, 어디에서 일어났는지 데이터에서 더 찾아보자.

  → ( )


⑦ `위치` 속성에서 첫 두 글자만 추출해 `지역` 속성에 저장한다.

**💡 참고: `위치`에서 `지역` 뽑아내기**

### 1. `위치` 열 선택하기

- **문법 설명:** `df['열 이름']`은 데이터 프레임에서 해당 열을 선택한다.
- **이번 활동:** 지진이 발생한 장소가 글자로 들어 있는 `위치` 열을 선택한다.

### 2. 글자 처리 기능에 접근하기

- **문법 설명:** 여러 행의 글자 데이터를 처리할 때는 열 뒤에 `.str`을 붙인다. `.str`을 통해 각 행의 글자에 같은 글자 처리를 적용할 수 있다.
- **사용 형식:** `글자가 들어 있는 열.str`
- **이번 활동:** 1번에서 선택한 `위치` 열 뒤에 `.str`을 붙인다.

### 3. 지역명과 뒤의 공백까지 뽑아내기

- **문법 설명:** 슬라이싱 `[ 시작 위치 : 끝 위치 ]`는 글자의 일부를 뽑아낸다. 시작 위치를 비우면 처음부터 시작하고, 끝 위치의 글자는 포함하지 않는다.

| 슬라이싱 | 의미 | `경북 경주시`에 적용한 결과 |
|:---:|:---|:---:|
| `[:1]` | 처음부터 1번 위치 전까지 | `경` |
| `[:2]` | 처음부터 2번 위치 전까지 | `경북` |
| `[:3]` | 처음부터 3번 위치 전까지 | `경북 ` |
| `[3:6]` | 3번 위치부터 6번 위치 전까지 | `경주시` |

- **이번 활동:** 교과서 코드는 두 글자인 지역명 뒤의 공백까지 포함해 처음 세 글자를 추출한다. 열의 처음부터 0~2번 위치까지 선택하도록 끝 값을 3으로 지정한다.

### 4. `지역` 열에 저장하기

- **문법 설명:** `df['저장할 열 이름'] = 계산한 결과`의 형태로 새 열을 만들 수 있다.
- **이번 활동:** 3번에서 뽑아낸 값을 `df['지역']`에 저장한다.

> **결과 예:** `경북 경주시 남남서쪽 8.7km 지역` → `경북`

⑧ 지역별 지진 발생 횟수를 원그래프로 시각화하여 지진이 가장 많이 발생한 지역을 살펴보자.

**💡 참고: 지역별 발생 횟수를 원그래프로 나타내기**

### 1. `지역` 열 선택하기

- **문법 설명:** `df['열 이름']`의 형태로 데이터 프레임의 특정 열을 선택한다.
- **이번 활동:** 앞에서 만든 `지역` 열을 선택한다.

### 2. 지역별 발생 횟수 구하기

- **API 설명:** `.value_counts()`는 열에 있는 각 값이 몇 번씩 나오는지 세어 준다. 결과에는 값의 이름과 각 값의 횟수가 함께 들어 있다.
- **사용 형식:** `선택한 열.value_counts()`
- **이번 활동:** 1번에서 선택한 열에 `.value_counts()`를 적용한다.

### 3. 발생 횟수를 `loc` 변수에 저장하기

- **문법 설명:** `변수 이름 = 계산한 결과`의 형태로 결과를 저장하면 뒤의 코드에서 다시 사용할 수 있다.
- **이번 활동:** 2번에서 구한 지역별 발생 횟수를 `loc`에 저장한다. 원그래프의 크기와 레이블을 지정할 때 이 변수를 사용한다.

### 4. `plt.pie()`로 원그래프 그리기

- **API 설명:** `plt.pie()`는 수치 데이터의 크기 비율을 원의 조각으로 나타낸다.
- **사용 형식:**

```text
plt.pie(수치 데이터, labels=표시할 이름, autopct=비율 표시 형식)
```

| 입력 | 문법·API 설명 | 이번 활동 지시 |
|:---:|:---|:---|
| 수치 데이터 | 원의 각 조각 크기를 결정한다. | 지역별 발생 횟수가 들어 있는 `loc`를 넣는다. |
| `labels` | 각 조각에 붙일 이름을 지정한다. `.index`는 빈도수 결과의 값 이름을 가져온다. | `loc.index`를 넣어 지역 이름을 표시한다. |
| `autopct` | 각 조각의 비율을 표시할 형식을 지정한다. | `"%1.1f%%"`를 넣어 소수점 첫째 자리까지 표시한다. |

### 5. 그래프 보여 주기

- **API 설명:** `plt.show()`는 만든 그래프를 화면에 표시한다.
- **이번 활동:** 원그래프를 만든 뒤 `plt.show()`를 실행한다.

In [ ]:
# ‘지역’ 속성 값의 빈도수 구하기
# 원그래프 시각화, labels : 레이블 지정, autopct : 값을 %로 표시

**❓ 질문**
- 우리나라에서 지진이 가장 많이 발생한 지역은 어디이고, 몇 %인가?

  → 지역 ( ) · ( ) %
- 그 지역에서 지진이 자주 발생하는 원인은 무엇일지 조사해 보자.

  → ( )


## 🏁 마무리: 스스로 정리하기

1. 이번 지진 데이터 분석에서, 펭귄 실습과 **똑같이 적용한 단계**는 무엇이었나?

   →

2. 지진 데이터에는 있었지만 펭귄 데이터에는 없던 처리 과정은 무엇이었나? (예: '시간'에서 연도 뽑기)

   →

3. 이 분석 결과로, "우리나라는 지진 안전지대인가?"라는 질문에 어떻게 답하겠는가?

   →
